In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [8]:
cols_grav = [
    # identifiers
    "iso3_o", "iso3_d", "year",
    
    # distance (lnD) — use harmonic, it's the most standard
    "distw_harmonic",
    
    # # language (Language variable)
    # "comlang_off",    # common official language
    # "comlang_ethno",  # spoken by 9%+ of population in both countries
    
    # optional but useful
    "contig",         # shared border (control variable)
    "comleg_posttrans", # common legal origin — relevant for your Legal variable
    "legal_new_o",   # legal system of the origin country
    "legal_new_d",   # legal system of the destination country
    "fta_wto",       # free trade agreement or WTO membership (control variable)
    
]

In [9]:
# read the post-2015 version
# Get the select columns from cols
df_post = pd.read_csv("../Raw/Gravity_csv_V202211/Gravity_V202211.csv", usecols=cols_grav)
df_post

,year,iso3_o,iso3_d,distw_harmonic,contig,legal_new_o,legal_new_d,comleg_posttrans,fta_wto
0,1948,ABW,ABW,NaN,NaN,NaN,NaN,NaN,NaN
1,1949,ABW,ABW,NaN,NaN,NaN,NaN,NaN,NaN
2,1950,ABW,ABW,NaN,NaN,NaN,NaN,NaN,NaN
3,1951,ABW,ABW,NaN,NaN,NaN,NaN,NaN,NaN
4,1952,ABW,ABW,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
4699291,2017,ZWE,ZWE,43.0,0.0,2.0,2.0,1.0,0.0
4699292,2018,ZWE,ZWE,43.0,0.0,2.0,2.0,1.0,0.0
4699293,2019,ZWE,ZWE,43.0,0.0,2.0,2.0,1.0,0.0
4699294,2020,ZWE,ZWE,42.0,0.0,2.0,2.0,1.0,0.0


In [10]:
df_post[["legal_new_o", "legal_new_d", "comleg_posttrans", "fta_wto"]].describe()

,legal_new_o,legal_new_d,comleg_posttrans,fta_wto
count,3.767400e+06,3.767400e+06,3.600524e+06,3.627844e+06
mean,1.618796e+00,1.618796e+00,3.511806e-01,3.833847e-02
std,7.741009e-01,7.741009e-01,4.773393e-01,1.920121e-01
min,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
50%,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
75%,2.000000e+00,2.000000e+00,1.000000e+00,0.000000e+00
max,5.000000e+00,5.000000e+00,1.000000e+00,1.000000e+00


In [11]:
df_post.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4699296 entries, 0 to 4699295
Data columns (total 9 columns):
 #   Column            Dtype  
---  ------            -----  
 0   year              int64  
 1   iso3_o            object 
 2   iso3_d            object 
 3   distw_harmonic    float64
 4   contig            float64
 5   legal_new_o       float64
 6   legal_new_d       float64
 7   comleg_posttrans  float64
 8   fta_wto           float64
dtypes: float64(6), int64(1), object(2)
memory usage: 322.7+ MB


In [12]:
cols_lang = ["iso_o", "iso_d", "cle", "csl", "col"]

In [13]:
# read language a well
df_lang = pd.read_stata("../Raw/cepii_ling_web.dta", convert_categoricals=False)
df_lang = df_lang[cols_lang].copy()
df_lang

,iso_o,iso_d,cle,csl,col
0,AFG,ALB,0.086628,0.0000,0
1,AFG,DZA,0.100619,0.0000,0
2,AFG,AND,0.141894,0.0000,0
3,AFG,AGO,0.140295,0.0000,0
4,AFG,AIA,0.109629,0.0000,0
...,...,...,...,...,...
37825,ZWE,VUT,0.129664,0.3528,1
37826,ZWE,VEN,0.051485,0.0000,0
37827,ZWE,VNM,0.025630,0.0000,0
37828,ZWE,YEM,0.095582,0.0000,0


In [14]:
# Rename columns to match iso column names
df_lang = df_lang.rename(columns={"iso_o": "iso3_o", "iso_d": "iso3_d"})

In [15]:
# Check intersection of iso3_o and iso3_d in both dataframes
combinations = set(zip(df_post["iso3_o"], df_post["iso3_d"])).intersection(set(zip(df_lang["iso3_o"], df_lang["iso3_d"])))
print(f"Number of matching combinations: {len(combinations)} out of {len(set(zip(df_post['iso3_o'], df_post['iso3_d'])))} in df_post and {len(set(zip(df_lang['iso3_o'], df_lang['iso3_d'])))} in df_lang")

Number of matching combinations: 37442 out of 59049 in df_post and 37830 in df_lang


In [16]:
# Join them together
df_gravity = df_post.merge(
    df_lang,
    left_on=["iso3_o", "iso3_d"],
    right_on=["iso3_o", "iso3_d"],
    how="left"
)
df_gravity

,year,iso3_o,iso3_d,distw_harmonic,contig,legal_new_o,legal_new_d,comleg_posttrans,fta_wto,cle,csl,col
0,1948,ABW,ABW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1949,ABW,ABW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1950,ABW,ABW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1951,ABW,ABW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1952,ABW,ABW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
4699291,2017,ZWE,ZWE,43.0,0.0,2.0,2.0,1.0,0.0,NaN,NaN,NaN
4699292,2018,ZWE,ZWE,43.0,0.0,2.0,2.0,1.0,0.0,NaN,NaN,NaN
4699293,2019,ZWE,ZWE,43.0,0.0,2.0,2.0,1.0,0.0,NaN,NaN,NaN
4699294,2020,ZWE,ZWE,42.0,0.0,2.0,2.0,1.0,0.0,NaN,NaN,NaN


In [17]:
df_gravity.describe().round(2)

,year,distw_harmonic,contig,legal_new_o,legal_new_d,comleg_posttrans,fta_wto,cle,csl,col
count,4699296.00,3627844.00,3627844.00,3767400.00,3767400.00,3600524.00,3627844.00,3003364.00,3003364.00,3003364.00
mean,1984.50,8602.49,0.01,1.62,1.62,0.35,0.04,0.13,0.12,0.15
std,21.36,4744.74,0.11,0.77,0.77,0.48,0.19,0.16,0.23,0.36
min,1948.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00
25%,1966.00,4845.00,0.00,1.00,1.00,0.00,0.00,0.04,0.00,0.00
50%,1984.50,8223.00,0.00,1.00,1.00,0.00,0.00,0.10,0.00,0.00
75%,2003.00,12181.00,0.00,2.00,2.00,1.00,0.00,0.15,0.11,0.00
max,2021.00,19904.00,1.00,5.00,5.00,1.00,1.00,1.00,1.00,1.00


In [19]:
df_gravity = df_gravity.rename(columns={
    "distw_harmonic":   "distance_km",
    "contig":           "shared_border",
    "comleg_posttrans": "common_legal_origin",
    "cle":              "language_proximity",
    "csl":              "language_spoken_share",
    "col":              "common_official_language",
    "legal_new_o":      "legal_origin_o",
    "legal_new_d":      "legal_origin_d",
    "fta_wto":          "fta_wto_membership",
})

In [20]:
# Get nans per column
df_gravity.isna().mean().sort_values(ascending=False).head(30)

language_proximity          0.360891
language_spoken_share       0.360891
common_official_language    0.360891
common_legal_origin         0.233816
distance_km                 0.228003
shared_border               0.228003
fta_wto_membership          0.228003
legal_origin_o              0.198305
legal_origin_d              0.198305
year                        0.000000
iso3_o                      0.000000
iso3_d                      0.000000
dtype: float64

In [21]:
# Save to clean data
path = "../Clean/cepii_gravity_language.csv"
df_gravity.to_csv(path, index=False)